# Generation Evaluation in RAG Systems
## (Procurement Context)

Let me break this down clearly. In a Retrieval-Augmented Generation (RAG) system for procurement, two components do very different jobs — and evaluating generation means knowing exactly what went wrong and where.

---

## What Retrieval Is Responsible For

Retrieval's job is to **find the right documents** and pass them to the LLM as context. It is not responsible for what the LLM does with them.

In procurement, retrieval should surface:
- The correct tender document (e.g., RFP-2024-IT-047)
- The right vendor's bid submission
- The applicable evaluation criteria or compliance checklist
- The correct contract clause or SLA section

**Example:** A user asks *"What is Vendor A's delivery timeline for the hardware tender?"* Retrieval should fetch Vendor A's bid document, specifically the section on delivery schedules. If it fetches Vendor B's document instead — that's a **retrieval failure**, not a generation failure.

> If the LLM gets bad documents, it will produce a bad answer. That's retrieval's fault, not the LLM's.

---

## What the LLM Is Responsible For

The LLM's job is to **read the retrieved context and generate a faithful, useful answer**. It should not add, invent, or contradict what the documents say.

The LLM is responsible for:
- Reading the retrieved passage correctly
- Synthesizing information across multiple documents
- Generating an answer in the right format, tone, and level of detail
- Staying within the bounds of what the context actually says

**Example:** Retrieval correctly surfaces Vendor A's bid, which states *"delivery within 45 business days of PO issuance."* The LLM is responsible for accurately extracting and presenting that fact — not paraphrasing it as *"approximately 2 months"* without qualification, and not fabricating a specific date that doesn't exist.

---

## What Hallucination Means

Hallucination is when the LLM **generates content that is not supported by — or directly contradicts — the retrieved documents**.

In procurement, this is high-stakes. Examples:

| Hallucination Type | Example |
|---|---|
| **Invented fact** | LLM says Vendor B offers a 3-year warranty, but their bid says 1 year |
| **Wrong number** | LLM quotes a unit price of ₹4,200 when the bid says ₹4,800 |
| **Fabricated clause** | LLM mentions a penalty clause that doesn't exist in the contract |
| **Wrong vendor** | LLM attributes Vendor C's compliance certificate to Vendor A |
| **Confident gap-filling** | Tender doc is silent on insurance; LLM says "insurance is included as standard" |

The last type is especially dangerous in procurement — **silence in a document is not the same as a yes**. A well-evaluated LLM should say *"The tender document does not specify insurance coverage"* rather than assuming.

---

## Answer Quality Beyond Factual Accuracy

Even if an answer is factually correct, it can still be a poor generation. Evaluation must cover:

### 1. Completeness
Did the LLM answer the *whole* question, or only part of it?

> **Question:** *"Does Vendor C meet all mandatory eligibility criteria?"*
>
> **Incomplete answer:** *"Vendor C has a valid GST registration."* ✗
>
> **Complete answer:** *"Vendor C meets 4 of 5 mandatory criteria. They hold valid GST registration, ISO 9001 certification, and have submitted their EMD of ₹2 lakhs. However, their annual turnover of ₹8 Cr falls below the required ₹10 Cr threshold — they do not meet criteria 3."* ✓

### 2. Tone
Is the tone appropriate for the procurement context?

Procurement outputs are often formal, legal-adjacent documents. The LLM should match that register.

> **Wrong tone:** *"Looks like Vendor A is actually a pretty solid choice here — their price is pretty decent!"* ✗
>
> **Right tone:** *"Vendor A's quoted price of ₹18.4 lakhs is 12% below the estimated budget and is competitive relative to other submissions."* ✓

### 3. Format
Is the output structured for how it will be used?

| Use Case | Expected Format |
|---|---|
| Bid comparison | Table with vendors as columns, criteria as rows |
| Compliance check | Checklist with pass/fail/missing per criterion |
| Summary for committee | Bullet-pointed executive summary |
| Contract clause lookup | Direct quote + section reference |

A factually correct answer dumped as a wall of prose when the evaluator needs a comparison table is still a **generation quality failure**.

### 4. Groundedness / Attribution
Does the answer tell you *where* the information came from?

> **Ungrounded:** *"The payment terms are net-60."*
>
> **Grounded:** *"Per Section 7.3 of the draft contract (RFP-2024-IT-047), payment terms are net-60 days from invoice date."* ✓

This matters enormously in procurement audits and dispute resolution.

---

## Summary

| Dimension | Who Owns It | Procurement Example |
|---|---|---|
| Wrong document fetched | **Retrieval** | Pulled last year's tender instead of current one |
| Invented fact | **LLM (hallucination)** | Made up a delivery date not in the bid |
| Partial answer | **LLM (completeness)** | Answered price but ignored warranty terms |
| Wrong register | **LLM (tone)** | Too casual for a committee evaluation report |
| Unstructured output | **LLM (format)** | Prose instead of a compliance checklist |
| No source reference | **LLM (attribution)** | Didn't cite which bid document the data came from |

Evaluating generation well means checking all six — not just whether the numbers are right.